# Week 2 — Data Cleaning, Integration & Validation
## Heavy Supplier, Inventory & Warehouse Analytics

---

### Objectives
1. Investigate the data-quality issues identified during Week 1.
2. Validate missing values, data types, identifiers, and relationships.
3. Define appropriate cleaning rules without losing valid business records.
4. Integrate related datasets while controlling join duplication.
5. Compare data-quality metrics before and after cleaning.
6. Document all changes and unresolved issues.

### Data Handling Principles
- **Preserve** the original raw datasets — never overwrite source CSVs.
- **Work on copies** — all transformations applied to in-memory working frames.
- **Never delete** records solely because an identifier is repeated.
- **Never invent** missing dates or quantities.
- **Record the reason** for every cleaning decision.
- **Validate** row counts and relationships after each transformation.

### Checklist (7 tasks)
| # | Task | Description |
|---|---|---|
| 1 | Load all 12 raw datasets | Confirm files and row counts match Week 1 |
| 2 | Review missing received dates | Check order status and document the cause |
| 3 | Investigate repeated invoice and payment IDs | Inspect complete records, confirm business rules |
| 4 | Validate candidate keys and relationships | Check uniqueness, nulls, and unmatched references |
| 5 | Reconcile inventory balances | Investigate product–branch balance mismatches |
| 6 | Integrate datasets safely | Check join cardinality, ensure financial totals not inflated |
| 7 | Save validation results and report | Record all results and unresolved issues |

---
## Setup — Imports & Directory Configuration

In [48]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_colwidth", 40)

# ── Paths ────────────────────────────────────────────────────────────────────
PROJECT_DIR  = Path.cwd().parent                        # heavy-supplier-analytics/
DATA_DIR     = PROJECT_DIR / "raw_data"                     # source CSVs  (read-only)
REPORTS_DIR  = PROJECT_DIR / "week_02" / "reports"      # all outputs go here
CLEAN_DIR    = REPORTS_DIR / "clean"                    # cleaned dataset exports

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Project  :", PROJECT_DIR.resolve())
print("Data     :", DATA_DIR.resolve())
print("Reports  :", REPORTS_DIR.resolve())
print("Clean    :", CLEAN_DIR.resolve())

# ── Audit log (every cleaning decision is appended here) ─────────────────────
audit_log: list[dict] = []

Project  : /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics
Data     : /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/raw_data
Reports  : /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week_02/reports
Clean    : /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week_02/reports/clean


---
## Task 1 — Load All 12 Raw Datasets
**Goal:** Confirm all CSV files are present and row counts exactly match the Week-1 baseline.

In [49]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")
for f in csv_files:
    print(" -", f.name)

# Load every file — low_memory=False for reliable type inference
raw: dict[str, pd.DataFrame] = {
    f.stem: pd.read_csv(f, low_memory=False)
    for f in csv_files
}

print(f"\nDatasets loaded: {len(raw)}")

CSV files found: 12
 - branches.csv
 - customers.csv
 - inventory_master.csv
 - invoices.csv
 - payments.csv
 - products.csv
 - purchase_orders_header.csv
 - purchase_orders_lines.csv
 - sales_orders_header.csv
 - sales_orders_lines.csv
 - stock_ledger.csv
 - suppliers.csv

Datasets loaded: 12


In [50]:
# ── Week-1 baseline counts (from executed notebook outputs) ──────────────────
WEEK1_COUNTS = {
    "branches"               :      6,
    "customers"              :    500,
    "inventory_master"       :    180,
    "invoices"               :  18_033,
    "payments"               :  19_257,
    "products"               :     30,
    "purchase_orders_header" :  24_000,
    "purchase_orders_lines"  : 155_495,
    "sales_orders_header"    :  20_000,
    "sales_orders_lines"     : 130_402,
    "stock_ledger"           : 237_230,
    "suppliers"              :      8,
}

comparison_rows = []
for name, expected in WEEK1_COUNTS.items():
    actual = len(raw[name]) if name in raw else None
    comparison_rows.append({
        "dataset"       : name,
        "week1_rows"    : expected,
        "current_rows"  : actual,
        "difference"    : (actual - expected) if actual is not None else None,
        "status"        : "✅ Match" if actual == expected else "❌ Mismatch",
    })

baseline_df = pd.DataFrame(comparison_rows)
display(baseline_df)

total_w1  = sum(WEEK1_COUNTS.values())
total_now = sum(len(raw[n]) for n in WEEK1_COUNTS)
print(f"\nWeek-1 total : {total_w1:,}")
print(f"Current total: {total_now:,}")
print(f"Difference   : {total_now - total_w1:,}")

# ── Save baseline comparison ──────────────────────────────────────────────────
baseline_df.to_csv(REPORTS_DIR / "task1_baseline_comparison.csv", index=False)
print("\nSaved: task1_baseline_comparison.csv")

audit_log.append({
    "task": "Task 1", "dataset": "all",
    "action": "Baseline row-count verification",
    "result": f"All {len(WEEK1_COUNTS)} datasets match Week-1 counts" if total_now == total_w1 else "Mismatch detected",
    "records_affected": 0, "decision": "No action — informational",
})

,dataset,week1_rows,current_rows,difference,status
0,branches,6,6,0,✅ Match
1,customers,500,500,0,✅ Match
2,inventory_master,180,180,0,✅ Match
3,invoices,18033,18033,0,✅ Match
4,payments,19257,19257,0,✅ Match
5,products,30,30,0,✅ Match
6,purchase_orders_header,24000,24000,0,✅ Match
7,purchase_orders_lines,155495,155495,0,✅ Match
8,sales_orders_header,20000,20000,0,✅ Match
9,sales_orders_lines,130402,130402,0,✅ Match



Week-1 total : 605,141
Current total: 605,141
Difference   : 0

Saved: task1_baseline_comparison.csv


In [51]:
# ── Quick overall profile ─────────────────────────────────────────────────────
profile_rows = []
for name, df in raw.items():
    profile_rows.append({
        "dataset"             : name,
        "rows"                : len(df),
        "columns"             : df.shape[1],
        "missing_cells"       : int(df.isna().sum().sum()),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "memory_mb"           : round(df.memory_usage(deep=True).sum() / 1_048_576, 2),
    })

profile_df = pd.DataFrame(profile_rows).sort_values("rows", ascending=False).reset_index(drop=True)
display(profile_df)

,dataset,rows,columns,missing_cells,exact_duplicate_rows,memory_mb
0,stock_ledger,237230,9,0,0,91.76
1,purchase_orders_lines,155495,9,0,0,24.76
2,sales_orders_lines,130402,9,0,0,20.77
3,purchase_orders_header,24000,10,2370,0,9.71
4,sales_orders_header,20000,11,0,0,9.19
5,payments,19257,5,0,0,4.43
6,invoices,18033,10,0,0,7.28
7,customers,500,15,0,0,0.29
8,inventory_master,180,8,0,0,0.03
9,products,30,23,0,0,0.02


---
## Task 2 — Review Missing Received Dates
**Goal:** Identify why 2,370 `received_date` values are null in `purchase_orders_header` and document the cause.

In [52]:
# Work on a copy — never mutate raw
po_hdr = raw["purchase_orders_header"].copy()

print("Columns:", po_hdr.columns.tolist())
print(f"Total rows         : {len(po_hdr):,}")
print(f"Missing received_date: {po_hdr['received_date'].isna().sum():,}")

# ── Cross-tab: po_status vs received_date presence ───────────────────────────
po_hdr_flagged = po_hdr.assign(date_missing=po_hdr["received_date"].isna())

status_cross = (
    po_hdr_flagged
    .groupby(["po_status", "date_missing"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "date_present", True: "date_missing"})
)
status_cross["total"]           = status_cross.sum(axis=1)
status_cross["missing_pct"]     = (status_cross["date_missing"] / status_cross["total"] * 100).round(2)

print("\nPO status vs received_date availability:")
display(status_cross)

Columns: ['po_id', 'supplier_id', 'branch_id', 'order_date', 'expected_delivery_date', 'received_date', 'po_status', 'total_cost', 'total_gst_amount', 'grand_total']
Total rows         : 24,000
Missing received_date: 2,370

PO status vs received_date availability:


date_missing,date_present,date_missing,total,missing_pct
po_status,,,,
Cancelled,0,2370,2370,100.00
Received,21630,0,21630,0.00


In [53]:
# ── Sample of cancelled POs with missing dates ───────────────────────────────
missing_rows = po_hdr[po_hdr["received_date"].isna()].copy()
print(f"All {len(missing_rows):,} missing-date rows belong to status:")
print(missing_rows["po_status"].value_counts().to_string())
print()
display(missing_rows[["po_id","supplier_id","branch_id",
                       "order_date","expected_delivery_date",
                       "po_status","grand_total"]].head(10))

All 2,370 missing-date rows belong to status:
po_status
Cancelled    2370



,po_id,supplier_id,branch_id,order_date,expected_delivery_date,po_status,grand_total
15,PO-525153,SUP0005,HYD001,2019-10-10,2019-11-07,Cancelled,"10,731,981.30"
20,PO-821311,SUP0006,HYD001,2020-04-15,2020-04-25,Cancelled,"553,847.65"
21,PO-171706,SUP0002,AHM001,2020-11-21,2020-12-01,Cancelled,"20,030,352.61"
24,PO-477755,SUP0004,AHM001,2020-12-14,2021-01-01,Cancelled,"13,008,692.38"
33,PO-908623,SUP0007,DEL001,2023-07-24,2023-08-05,Cancelled,"15,968,445.39"
37,PO-409171,SUP0008,AHM001,2020-04-12,2020-05-08,Cancelled,"17,635,289.25"
40,PO-846758,SUP0004,CHN001,2022-10-21,2022-11-08,Cancelled,"13,739,169.44"
41,PO-318205,SUP0007,HYD001,2020-11-26,2020-12-08,Cancelled,"15,749,643.14"
58,PO-825717,SUP0002,AHM001,2024-07-20,2024-07-30,Cancelled,"37,879,677.86"
59,PO-476858,SUP0004,PUN001,2020-09-22,2020-10-10,Cancelled,"36,870,871.59"


In [54]:
# ── Cleaning decision ─────────────────────────────────────────────────────────
# FINDING   : Every missing received_date belongs to po_status = 'Cancelled'.
# CAUSE     : Cancelled orders are never received, so no received_date exists.
# DECISION  : Retain all 2,370 rows. No imputation. Null is expected for Cancelled status.
# VALIDATION: Zero Received-status POs have a missing date → rule is clean.

received_null_in_received_status = (
    po_hdr[
        (po_hdr["po_status"] == "Received") &
        (po_hdr["received_date"].isna())
    ]
)
print(f"Received-status POs with missing date: {len(received_null_in_received_status)} (expected 0)")

# Save investigation
status_cross.to_csv(REPORTS_DIR / "task2_received_date_status_analysis.csv")
print("Saved: task2_received_date_status_analysis.csv")

audit_log.append({
    "task"             : "Task 2",
    "dataset"          : "purchase_orders_header",
    "action"           : "Investigated missing received_date",
    "result"           : "100% of nulls belong to Cancelled POs; 0 Received POs have null",
    "records_affected" : 2_370,
    "decision"         : "Retain all rows. Null received_date is valid for Cancelled status.",
})

Received-status POs with missing date: 0 (expected 0)
Saved: task2_received_date_status_analysis.csv


---
## Task 3 — Investigate Repeated Invoice & Payment IDs
**Goal:** Inspect every repeated identifier, confirm whether records are genuine duplicates or different transactions sharing a bad ID, and define a cleaning rule.

### 3-A  Repeated `invoice_id`

In [55]:
inv = raw["invoices"].copy()

id_freq = inv["invoice_id"].value_counts()
repeated_inv_ids = id_freq[id_freq > 1]

print(f"Total invoice rows       : {len(inv):,}")
print(f"Distinct invoice_id      : {inv['invoice_id'].nunique():,}")
print(f"Repeated invoice_id count: {len(repeated_inv_ids):,}")
print(f"Rows with repeated ID    : {int(repeated_inv_ids.sum()):,}")
print(f"Extra occurrences        : {int((repeated_inv_ids - 1).sum()):,}")

Total invoice rows       : 18,033
Distinct invoice_id      : 17,836
Repeated invoice_id count: 196
Rows with repeated ID    : 393
Extra occurrences        : 197


In [56]:
# ── Per-ID diversity: are the repeated rows identical or distinct transactions? ─
inv_dup_detail = (
    inv.groupby("invoice_id", sort=False)
    .agg(
        row_count              = ("invoice_id",        "size"),
        distinct_so_ids        = ("so_id",             "nunique"),
        distinct_customers     = ("customer_id",       "nunique"),
        distinct_branches      = ("branch_id",         "nunique"),
        distinct_invoice_dates = ("invoice_date",      "nunique"),
        distinct_grand_totals  = ("grand_total",       "nunique"),
        distinct_pay_status    = ("payment_status",    "nunique"),
    )
    .reset_index()
)

inv_dup_detail = inv_dup_detail[inv_dup_detail["row_count"] > 1].sort_values(
    "row_count", ascending=False
).reset_index(drop=True)

# Are ANY repeated rows perfectly identical (all fields the same)?
fully_identical_inv = inv_dup_detail[
    (inv_dup_detail["distinct_so_ids"]        == 1) &
    (inv_dup_detail["distinct_customers"]      == 1) &
    (inv_dup_detail["distinct_invoice_dates"]  == 1) &
    (inv_dup_detail["distinct_grand_totals"]   == 1)
]

print(f"Repeated invoice IDs           : {len(inv_dup_detail)}")
print(f"All repeated with distinct SO  : {(inv_dup_detail['distinct_so_ids'] > 1).sum()}")
print(f"All repeated with distinct date: {(inv_dup_detail['distinct_invoice_dates'] > 1).sum()}")
print(f"Fully identical duplicates     : {len(fully_identical_inv)}")
print()
display(inv_dup_detail.head(20))

Repeated invoice IDs           : 196
All repeated with distinct SO  : 196
All repeated with distinct date: 196
Fully identical duplicates     : 0



,invoice_id,row_count,distinct_so_ids,distinct_customers,distinct_branches,distinct_invoice_dates,distinct_grand_totals,distinct_pay_status
0,INV-341709,3,3,3,3,3,3,1
1,INV-361583,2,2,2,1,2,2,1
2,INV-489554,2,2,2,2,2,2,2
3,INV-166804,2,2,2,2,2,2,2
4,INV-632936,2,2,2,2,2,2,1
5,INV-695723,2,2,2,2,2,2,1
6,INV-414608,2,2,2,2,2,2,1
7,INV-326629,2,2,2,2,2,2,2
8,INV-850022,2,2,2,1,2,2,1
9,INV-667205,2,2,2,1,2,2,2


In [57]:
# ── Deep-dive on the ID that appears 3 times ─────────────────────────────────
triple_id = inv_dup_detail[inv_dup_detail["row_count"] == 3]["invoice_id"].iloc[0]
print(f"Invoice ID appearing 3 times: {triple_id}")
display(
    inv[inv["invoice_id"] == triple_id]
    .sort_values("invoice_date")
    .reset_index(drop=True)
)

Invoice ID appearing 3 times: INV-341709


,invoice_id,so_id,customer_id,branch_id,invoice_date,due_date,total_order_value,total_gst_amount,grand_total,payment_status
0,INV-341709,SO-533236,C0345,PUN001,2021-07-10,2021-08-09,803850,"222,313.00","1,026,163.00",Paid
1,INV-341709,SO-455910,C0078,HYD001,2022-01-18,2022-02-17,2281250,"606,975.00","2,888,225.00",Paid
2,INV-341709,SO-510296,C0176,CHN001,2024-08-11,2024-08-26,551950,"136,641.00","688,591.00",Paid


In [58]:
# ── Cleaning decision for invoices ───────────────────────────────────────────
# FINDING : 196 invoice_id values appear more than once.
#           Every repeated ID links to DIFFERENT so_id, customer, date, and amount.
#           Fully-identical duplicates = 0.
# CAUSE   : Invoice numbering defect in source system; the same numeric ID
#           was issued to unrelated transactions.
# DECISION: Retain all rows.  Add a surrogate row key and a flag column.
#           Do NOT remove rows — each represents a distinct business transaction.

inv_clean = inv.copy()
inv_clean.insert(0, "row_key", range(1, len(inv_clean) + 1))
inv_clean["duplicate_invoice_id_flag"] = inv_clean["invoice_id"].isin(
    set(inv_dup_detail["invoice_id"])
)

flagged = inv_clean["duplicate_invoice_id_flag"].sum()
print(f"Rows flagged as duplicate_invoice_id: {flagged} / {len(inv_clean):,}")

# Save detail report
inv_dup_detail.to_csv(REPORTS_DIR / "task3a_invoice_id_duplicate_analysis.csv", index=False)

audit_log.append({
    "task"             : "Task 3-A",
    "dataset"          : "invoices",
    "action"           : "Investigated 196 repeated invoice_id values",
    "result"           : "All repeated IDs link to distinct transactions. 0 fully identical rows.",
    "records_affected" : int(flagged),
    "decision"         : "Retain all rows. Added row_key surrogate and duplicate_invoice_id_flag.",
})

Rows flagged as duplicate_invoice_id: 393 / 18,033


### 3-B  Repeated `payment_id`

In [59]:
pay = raw["payments"].copy()

pay_freq = pay["payment_id"].value_counts()
repeated_pay_ids = pay_freq[pay_freq > 1]

print(f"Total payment rows       : {len(pay):,}")
print(f"Distinct payment_id      : {pay['payment_id'].nunique():,}")
print(f"Repeated payment_id count: {len(repeated_pay_ids):,}")
print(f"Rows with repeated ID    : {int(repeated_pay_ids.sum()):,}")
print(f"Extra occurrences        : {int((repeated_pay_ids - 1).sum()):,}")

Total payment rows       : 19,257
Distinct payment_id      : 19,055
Repeated payment_id count: 201
Rows with repeated ID    : 403
Extra occurrences        : 202


In [60]:
pay_dup_detail = (
    pay.groupby("payment_id", sort=False)
    .agg(
        row_count             = ("payment_id",     "size"),
        distinct_invoices     = ("invoice_id",     "nunique"),
        distinct_dates        = ("payment_date",   "nunique"),
        distinct_amounts      = ("payment_amount", "nunique"),
        distinct_methods      = ("payment_method", "nunique"),
    )
    .reset_index()
)
pay_dup_detail = pay_dup_detail[pay_dup_detail["row_count"] > 1].sort_values(
    "row_count", ascending=False
).reset_index(drop=True)

fully_identical_pay = pay_dup_detail[
    (pay_dup_detail["distinct_invoices"] == 1) &
    (pay_dup_detail["distinct_dates"]    == 1) &
    (pay_dup_detail["distinct_amounts"]  == 1)
]

print(f"Repeated payment IDs              : {len(pay_dup_detail)}")
print(f"With distinct invoice references  : {(pay_dup_detail['distinct_invoices'] > 1).sum()}")
print(f"With distinct amounts             : {(pay_dup_detail['distinct_amounts']  > 1).sum()}")
print(f"Fully identical duplicates        : {len(fully_identical_pay)}")
print()
display(pay_dup_detail.head(20))

Repeated payment IDs              : 201
With distinct invoice references  : 201
With distinct amounts             : 201
Fully identical duplicates        : 0



,payment_id,row_count,distinct_invoices,distinct_dates,distinct_amounts,distinct_methods
0,PAY-361416,3,3,3,3,2
1,PAY-761769,2,2,2,2,2
2,PAY-414584,2,2,2,2,2
3,PAY-882840,2,2,2,2,2
4,PAY-524001,2,2,2,2,2
5,PAY-899223,2,2,2,2,2
6,PAY-821281,2,2,2,2,2
7,PAY-624298,2,2,2,2,2
8,PAY-372535,2,2,2,2,2
9,PAY-841086,2,2,2,2,2


In [61]:
# ── Cleaning decision for payments ───────────────────────────────────────────
# FINDING : 201 payment_id values appear more than once.
#           All repeated IDs link to different invoice references, dates, amounts.
#           Fully-identical duplicates = 0.
# CAUSE   : Payment ID generation defect — same ID issued to separate transactions.
# DECISION: Retain all rows.  Add surrogate row key and flag column.

pay_clean = pay.copy()
pay_clean.insert(0, "row_key", range(1, len(pay_clean) + 1))
pay_clean["duplicate_payment_id_flag"] = pay_clean["payment_id"].isin(
    set(pay_dup_detail["payment_id"])
)

flagged_pay = pay_clean["duplicate_payment_id_flag"].sum()
print(f"Rows flagged as duplicate_payment_id: {flagged_pay} / {len(pay_clean):,}")

pay_dup_detail.to_csv(REPORTS_DIR / "task3b_payment_id_duplicate_analysis.csv", index=False)

audit_log.append({
    "task"             : "Task 3-B",
    "dataset"          : "payments",
    "action"           : "Investigated 201 repeated payment_id values",
    "result"           : "All repeated IDs link to distinct transactions. 0 fully identical rows.",
    "records_affected" : int(flagged_pay),
    "decision"         : "Retain all rows. Added row_key surrogate and duplicate_payment_id_flag.",
})

Rows flagged as duplicate_payment_id: 403 / 19,257


---
## Task 4 — Validate Candidate Keys & Relationships
**Goal:** For each candidate primary key confirm uniqueness and non-null. For each FK relationship confirm all child values exist in the parent table.

In [62]:
# ── 4-A  Candidate primary-key uniqueness & null checks ──────────────────────
PK_CHECKS = [
    ("branches",               "branch_id"),
    ("customers",              "customer_id"),
    ("products",               "product_id"),
    ("suppliers",              "supplier_id"),
    ("purchase_orders_header", "po_id"),
    ("sales_orders_header",    "so_id"),
]

pk_results = []
for tbl, col in PK_CHECKS:
    df = raw[tbl]
    n      = len(df)
    nulls  = int(df[col].isna().sum())
    unique = int(df[col].nunique(dropna=True))
    dupes  = n - unique - nulls
    pk_results.append({
        "dataset"       : tbl,
        "key_column"    : col,
        "total_rows"    : n,
        "null_values"   : nulls,
        "distinct_values": unique,
        "duplicate_keys": dupes,
        "key_valid"     : nulls == 0 and dupes == 0,
    })

pk_df = pd.DataFrame(pk_results)
pk_df["status"] = pk_df["key_valid"].map({True: "✅ Valid", False: "❌ Invalid"})
display(pk_df)

pk_df.to_csv(REPORTS_DIR / "task4a_primary_key_validation.csv", index=False)
print("Saved: task4a_primary_key_validation.csv")

,dataset,key_column,total_rows,null_values,distinct_values,duplicate_keys,key_valid,status
0,branches,branch_id,6,0,6,0,True,✅ Valid
1,customers,customer_id,500,0,500,0,True,✅ Valid
2,products,product_id,30,0,30,0,True,✅ Valid
3,suppliers,supplier_id,8,0,8,0,True,✅ Valid
4,purchase_orders_header,po_id,24000,0,24000,0,True,✅ Valid
5,sales_orders_header,so_id,20000,0,20000,0,True,✅ Valid


Saved: task4a_primary_key_validation.csv


In [63]:
# ── 4-B  Foreign-key referential integrity ────────────────────────────────────
FK_CHECKS = [
    # (child_table, child_col, parent_table, parent_col)
    ("customers",               "branch_id",   "branches",               "branch_id"),
    ("inventory_master",        "product_id",  "products",               "product_id"),
    ("inventory_master",        "branch_id",   "branches",               "branch_id"),
    ("purchase_orders_header",  "supplier_id", "suppliers",              "supplier_id"),
    ("purchase_orders_header",  "branch_id",   "branches",               "branch_id"),
    ("purchase_orders_lines",   "po_id",       "purchase_orders_header", "po_id"),
    ("purchase_orders_lines",   "product_id",  "products",               "product_id"),
    ("sales_orders_header",     "customer_id", "customers",              "customer_id"),
    ("sales_orders_header",     "branch_id",   "branches",               "branch_id"),
    ("sales_orders_lines",      "so_id",       "sales_orders_header",    "so_id"),
    ("sales_orders_lines",      "product_id",  "products",               "product_id"),
    ("invoices",                "so_id",       "sales_orders_header",    "so_id"),
    ("invoices",                "customer_id", "customers",              "customer_id"),
    ("invoices",                "branch_id",   "branches",               "branch_id"),
    ("payments",                "invoice_id",  "invoices",               "invoice_id"),
    ("stock_ledger",            "product_id",  "products",               "product_id"),
    ("stock_ledger",            "branch_id",   "branches",               "branch_id"),
]

fk_results = []
for child_tbl, child_col, parent_tbl, parent_col in FK_CHECKS:
    child_vals  = raw[child_tbl][child_col].dropna()
    parent_keys = set(raw[parent_tbl][parent_col].dropna().unique())
    unmatched   = child_vals[~child_vals.isin(parent_keys)]
    null_count  = int(raw[child_tbl][child_col].isna().sum())
    fk_results.append({
        "child_dataset"    : child_tbl,
        "child_column"     : child_col,
        "parent_dataset"   : parent_tbl,
        "parent_column"    : parent_col,
        "child_total_rows" : len(raw[child_tbl]),
        "null_in_child"    : null_count,
        "unmatched_refs"   : len(unmatched),
        "status"           : "✅ Pass" if len(unmatched) == 0 else "❌ Fail",
    })

fk_df = pd.DataFrame(fk_results)
display(fk_df)

fk_df.to_csv(REPORTS_DIR / "task4b_foreign_key_validation.csv", index=False)
print(f"\nRelationships passing: {(fk_df['status'] == '✅ Pass').sum()} / {len(fk_df)}")
print("Saved: task4b_foreign_key_validation.csv")

audit_log.append({
    "task"             : "Task 4",
    "dataset"          : "multiple",
    "action"           : "Validated 6 candidate primary keys + 17 FK relationships",
    "result"           : f"{(pk_df['key_valid']).sum()}/6 PKs valid; {(fk_df['status']=='✅ Pass').sum()}/{len(fk_df)} FKs pass",
    "records_affected" : 0,
    "decision"         : "No deletions. All failures documented for business-rule confirmation.",
})

,child_dataset,child_column,parent_dataset,parent_column,child_total_rows,null_in_child,unmatched_refs,status
0,customers,branch_id,branches,branch_id,500,0,0,✅ Pass
1,inventory_master,product_id,products,product_id,180,0,0,✅ Pass
2,inventory_master,branch_id,branches,branch_id,180,0,0,✅ Pass
3,purchase_orders_header,supplier_id,suppliers,supplier_id,24000,0,0,✅ Pass
4,purchase_orders_header,branch_id,branches,branch_id,24000,0,0,✅ Pass
5,purchase_orders_lines,po_id,purchase_orders_header,po_id,155495,0,0,✅ Pass
6,purchase_orders_lines,product_id,products,product_id,155495,0,0,✅ Pass
7,sales_orders_header,customer_id,customers,customer_id,20000,0,0,✅ Pass
8,sales_orders_header,branch_id,branches,branch_id,20000,0,0,✅ Pass
9,sales_orders_lines,so_id,sales_orders_header,so_id,130402,0,0,✅ Pass



Relationships passing: 17 / 17
Saved: task4b_foreign_key_validation.csv


---
## Task 5 — Reconcile Inventory Balances
**Goal:** Compare `inventory_master.current_stock` with the latest `running_balance` in `stock_ledger` for each `(product_id, branch_id)` combination. Identify and investigate mismatches.

In [64]:
ledger = raw["stock_ledger"].copy()
inv_master = raw["inventory_master"].copy()

print("Stock ledger columns :", ledger.columns.tolist())
print("Inventory master cols:", inv_master.columns.tolist())
print(f"\nLedger rows : {len(ledger):,}")
print(f"Master rows : {len(inv_master):,}")
print(f"\nMovement types:\n{ledger['movement_type'].value_counts().to_string()}")

Stock ledger columns : ['movement_id', 'product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'reference_type', 'reference_id', 'running_balance']
Inventory master cols: ['product_id', 'branch_id', 'opening_stock', 'reorder_level', 'safety_stock', 'max_stock', 'current_stock', 'warehouse_bin']

Ledger rows : 237,230
Master rows : 180

Movement types:
movement_type
IN            127611
OUT           107165
ADJUSTMENT      2454


In [65]:
# ── Parse movement_date and take the latest running_balance per (product, branch)
ledger["movement_date"] = pd.to_datetime(ledger["movement_date"], errors="coerce")

# FIX: movement_id is NOT chronological (e.g. MOV-IN-79354 comes after MOV-OUT-184742),
# so it must not be used as a same-day tie-break. The file order of the ledger is the
# true posting order (running_balance chains correctly in that order), so use it.
ledger["ledger_pos"] = np.arange(len(ledger))
ledger_sorted = ledger.sort_values(
    ["product_id", "branch_id", "movement_date", "ledger_pos"],
    ascending=[True, True, False, False],         # latest date / latest posting first
    kind="stable",
)

latest_balance = (
    ledger_sorted
    .groupby(["product_id", "branch_id"], sort=False)
    .agg(
        ledger_latest_balance = ("running_balance", "first"),
        latest_movement_date  = ("movement_date",   "first"),
        total_movements       = ("movement_id",      "count"),
    )
    .reset_index()
)

print(f"Unique (product_id, branch_id) pairs in ledger : {len(latest_balance):,}")
print(f"Unique (product_id, branch_id) pairs in master : {len(inv_master):,}")
display(latest_balance.head(5))

Unique (product_id, branch_id) pairs in ledger : 180
Unique (product_id, branch_id) pairs in master : 180


,product_id,branch_id,ledger_latest_balance,latest_movement_date,total_movements
0,P001,AHM001,106714,2025-01-27,1292
1,P001,CHN001,99373,2025-01-16,1426
2,P001,DEL001,115928,2025-01-19,1311
3,P001,HYD001,108772,2025-01-27,1220
4,P001,KOL001,103808,2025-01-18,1372


In [66]:
# ── Merge and compare ─────────────────────────────────────────────────────────
reconcile = inv_master.merge(
    latest_balance,
    on=["product_id", "branch_id"],
    how="left"
)

reconcile["balance_difference"] = (
    reconcile["current_stock"] - reconcile["ledger_latest_balance"]
)
reconcile["mismatch"] = reconcile["balance_difference"] != 0

n_mismatches = reconcile["mismatch"].sum()
print(f"Total (product, branch) pairs checked : {len(reconcile):,}")
print(f"Pairs where current_stock == ledger   : {(~reconcile['mismatch']).sum()}")
print(f"Pairs where current_stock != ledger   : {n_mismatches}   ← mismatches")

Total (product, branch) pairs checked : 180
Pairs where current_stock == ledger   : 166
Pairs where current_stock != ledger   : 14   ← mismatches


In [67]:
# ── Detail for mismatched rows ────────────────────────────────────────────────
mismatch_rows = reconcile[reconcile["mismatch"]].copy().reset_index(drop=True)

display(
    mismatch_rows[
        ["product_id","branch_id","current_stock",
         "ledger_latest_balance","balance_difference",
         "latest_movement_date","total_movements"]
    ]
)

print(f"\nBalance difference stats (mismatched pairs):")
print(mismatch_rows["balance_difference"].describe().to_string())

# ── Investigate one mismatch in detail ───────────────────────────────────────
if len(mismatch_rows) > 0:
    sample = mismatch_rows.iloc[0]
    pid, bid = sample["product_id"], sample["branch_id"]
    print(f"\nLedger history for ({pid}, {bid}) — last 10 movements:")
    display(
        ledger[
            (ledger["product_id"] == pid) &
            (ledger["branch_id"]  == bid)
        ]
        .sort_values("movement_date", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

,product_id,branch_id,current_stock,ledger_latest_balance,balance_difference,latest_movement_date,total_movements
0,P002,CHN001,105885,106182,-297,2025-01-20,1462
1,P004,PUN001,104859,105024,-165,2025-01-18,1242
2,P008,AHM001,109281,109317,-36,2025-01-24,1350
3,P010,AHM001,115149,115549,-400,2025-01-24,1400
4,P021,DEL001,115902,116107,-205,2025-01-17,1258
5,P023,HYD001,105798,105981,-183,2025-01-15,1134
6,P025,HYD001,113613,114128,-515,2025-01-13,1271
7,P026,PUN001,105190,105335,-145,2025-01-14,1230
8,P026,HYD001,108348,108546,-198,2025-01-18,1264
9,P028,DEL001,106744,106967,-223,2025-01-23,1298



Balance difference stats (mismatched pairs):
count     14.00
mean    -194.86
std      142.23
min     -515.00
25%     -235.00
50%     -190.50
75%      -96.25
max       16.00

Ledger history for (P002, CHN001) — last 10 movements:


,movement_id,product_id,branch_id,movement_type,movement_date,quantity,reference_type,reference_id,running_balance,ledger_pos
0,MOV-IN-114717,P002,CHN001,IN,2025-01-20,297,PO,PO-148144,106182,10713
1,MOV-IN-114715,P002,CHN001,IN,2025-01-20,81,PO,PO-148144,105885,10712
2,MOV-IN-81488,P002,CHN001,IN,2025-01-17,201,PO,PO-557486,105804,10711
3,MOV-IN-70596,P002,CHN001,IN,2025-01-14,60,PO,PO-753393,105603,10710
4,MOV-IN-137480,P002,CHN001,IN,2025-01-14,298,PO,PO-503060,105543,10709
5,MOV-IN-69850,P002,CHN001,IN,2025-01-11,202,PO,PO-583134,105245,10708
6,MOV-OUT-151817,P002,CHN001,OUT,2025-01-10,4,SO,SO-283662,105043,10707
7,MOV-OUT-151815,P002,CHN001,OUT,2025-01-10,12,SO,SO-283662,105047,10706
8,MOV-IN-155365,P002,CHN001,IN,2025-01-05,37,PO,PO-195889,105059,10705
9,MOV-IN-58204,P002,CHN001,IN,2025-01-04,62,PO,PO-646167,105022,10704


In [68]:
# ── Save reconciliation results ───────────────────────────────────────────────
reconcile_cols = [
    "product_id","branch_id","current_stock",
    "ledger_latest_balance","balance_difference",
    "mismatch","latest_movement_date","total_movements"
]
reconcile[reconcile_cols].to_csv(
    REPORTS_DIR / "task5_inventory_balance_reconciliation.csv", index=False
)
mismatch_rows[reconcile_cols].to_csv(
    REPORTS_DIR / "task5_inventory_balance_mismatches.csv", index=False
)
print(f"Saved: task5_inventory_balance_reconciliation.csv ({len(reconcile)} rows)")
print(f"Saved: task5_inventory_balance_mismatches.csv ({n_mismatches} rows)")

# ── Composite key validation ──────────────────────────────────────────────────
n_products = inv_master["product_id"].nunique()
n_branches = inv_master["branch_id"].nunique()
comp_dupes = int(inv_master.duplicated(subset=["product_id","branch_id"]).sum())
print(f"\nInventory master composite key (product_id, branch_id):")
print(f"  Distinct products : {n_products}")
print(f"  Distinct branches : {n_branches}")
print(f"  Expected rows     : {n_products * n_branches}")
print(f"  Actual rows       : {len(inv_master)}")
print(f"  Composite dupes   : {comp_dupes}")
print(f"  Key valid         : {comp_dupes == 0 and len(inv_master) == n_products * n_branches}")

audit_log.append({
    "task"             : "Task 5",
    "dataset"          : "inventory_master + stock_ledger",
    "action"           : "Reconciled current_stock vs latest ledger running_balance",
    "result"           : f"{n_mismatches} mismatches found out of {len(reconcile)} pairs",
    "records_affected" : int(n_mismatches),
    "decision"         : "Mismatches documented. No correction applied — requires business sign-off on authoritative balance source.",
})

Saved: task5_inventory_balance_reconciliation.csv (180 rows)
Saved: task5_inventory_balance_mismatches.csv (14 rows)

Inventory master composite key (product_id, branch_id):
  Distinct products : 30
  Distinct branches : 6
  Expected rows     : 180
  Actual rows       : 180
  Composite dupes   : 0
  Key valid         : True


---
## Task 6 — Integrate Datasets Safely
**Goal:** Join related tables while verifying join cardinality at every step so financial totals are not inflated by fan-out.

### 6-A  Sales Orders: Header ↔ Lines reconciliation

In [69]:
so_hdr   = raw["sales_orders_header"].copy()
so_lines = raw["sales_orders_lines"].copy()

# Parse dates
for col in ["order_date","delivery_date"]:
    so_hdr[col] = pd.to_datetime(so_hdr[col], errors="coerce")

# ── Cardinality check before join ─────────────────────────────────────────────
lines_per_so = so_lines.groupby("so_id")["line_number"].count()
print("Lines per SO order — distribution:")
print(lines_per_so.describe().to_string())
print()

# SOs in header vs SOs referenced in lines
hdr_so_ids   = set(so_hdr["so_id"])
lines_so_ids = set(so_lines["so_id"])
print(f"SOs in header        : {len(hdr_so_ids):,}")
print(f"SOs referenced in lines: {len(lines_so_ids):,}")
print(f"In header NOT in lines: {len(hdr_so_ids - lines_so_ids)}")
print(f"In lines NOT in header: {len(lines_so_ids - hdr_so_ids)}")

Lines per SO order — distribution:
count   20,000.00
mean         6.52
std          3.45
min          1.00
25%          4.00
50%          7.00
75%         10.00
max         12.00

SOs in header        : 20,000
SOs referenced in lines: 20,000
In header NOT in lines: 0
In lines NOT in header: 0


In [70]:
# ── Aggregate lines to order level ───────────────────────────────────────────
# PRINCIPLE: Always aggregate the 'many' side before joining to the 'one' side
#            to avoid row multiplication.

lines_agg = (
    so_lines
    .groupby("so_id", sort=False)
    .agg(
        line_count                  = ("line_number",      "count"),
        lines_total_order_value     = ("line_total",       "sum"),
        lines_total_gst             = ("gst_amount",       "sum"),
        lines_grand_total           = ("line_grand_total", "sum"),
        distinct_products           = ("product_id",       "nunique"),
        total_quantity              = ("quantity",         "sum"),
    )
    .reset_index()
)

# ── Join: aggregated lines onto header (now a 1-to-1 join) ───────────────────
so_integrated = so_hdr.merge(lines_agg, on="so_id", how="left")

assert len(so_integrated) == len(so_hdr), "Row count changed after join — cardinality error!"
print(f"Header rows before join : {len(so_hdr):,}")
print(f"Header rows after join  : {len(so_integrated):,}  ✅ No fan-out")

Header rows before join : 20,000
Header rows after join  : 20,000  ✅ No fan-out


In [71]:
# ── Validate: header grand_total vs lines-aggregated grand_total ──────────────
so_integrated["grand_total_diff"] = (
    so_integrated["grand_total"] - so_integrated["lines_grand_total"]
).round(2)

# Allow ±1 rounding tolerance
so_integrated["totals_match"] = so_integrated["grand_total_diff"].abs() <= 1.0

match_count    = so_integrated["totals_match"].sum()
mismatch_count = (~so_integrated["totals_match"]).sum()
null_lines     = so_integrated["lines_grand_total"].isna().sum()

print(f"Sales orders with totals matching (±₹1) : {match_count:,}")
print(f"Sales orders with totals mismatching    : {mismatch_count:,}")
print(f"Sales orders with no line data          : {null_lines:,}")

if mismatch_count > 0:
    print("\nSample mismatches:")
    display(
        so_integrated[~so_integrated["totals_match"]][
            ["so_id","grand_total","lines_grand_total","grand_total_diff"]
        ].head(10)
    )

so_val_summary = pd.DataFrame([{
    "check": "SO header grand_total vs lines sum (±₹1)",
    "total_orders"  : len(so_integrated),
    "match"         : int(match_count),
    "mismatch"      : int(mismatch_count),
    "no_line_data"  : int(null_lines),
    "status"        : "✅ Pass" if mismatch_count == 0 else "⚠️ Investigate",
}])
display(so_val_summary)

Sales orders with totals matching (±₹1) : 20,000
Sales orders with totals mismatching    : 0
Sales orders with no line data          : 0


,check,total_orders,match,mismatch,no_line_data,status
0,SO header grand_total vs lines sum (...,20000,20000,0,0,✅ Pass


### 6-B  Purchase Orders: Header ↔ Lines reconciliation

In [72]:
po_hdr_clean  = raw["purchase_orders_header"].copy()
po_lines      = raw["purchase_orders_lines"].copy()

for col in ["order_date","expected_delivery_date","received_date"]:
    po_hdr_clean[col] = pd.to_datetime(po_hdr_clean[col], errors="coerce")

po_lines_agg = (
    po_lines
    .groupby("po_id", sort=False)
    .agg(
        line_count              = ("line_number",      "count"),
        lines_total_cost        = ("line_total",       "sum"),
        lines_total_gst         = ("gst_amount",       "sum"),
        lines_grand_total       = ("line_grand_total", "sum"),
        distinct_products       = ("product_id",       "nunique"),
        total_quantity_ordered  = ("quantity",         "sum"),
    )
    .reset_index()
)

po_integrated = po_hdr_clean.merge(po_lines_agg, on="po_id", how="left")

assert len(po_integrated) == len(po_hdr_clean), "PO cardinality error!"
print(f"PO header rows before join : {len(po_hdr_clean):,}")
print(f"PO header rows after join  : {len(po_integrated):,}  ✅ No fan-out")

po_integrated["grand_total_diff"] = (
    po_integrated["grand_total"] - po_integrated["lines_grand_total"]
).round(2)
po_integrated["totals_match"] = po_integrated["grand_total_diff"].abs() <= 1.0

po_match    = po_integrated["totals_match"].sum()
po_mismatch = (~po_integrated["totals_match"]).sum()
print(f"\nPO totals matching (±₹1)  : {po_match:,}")
print(f"PO totals mismatching     : {po_mismatch:,}")

po_val_summary = pd.DataFrame([{
    "check"        : "PO header grand_total vs lines sum (±₹1)",
    "total_orders" : len(po_integrated),
    "match"        : int(po_match),
    "mismatch"     : int(po_mismatch),
    "status"       : "✅ Pass" if po_mismatch == 0 else "⚠️ Investigate",
}])
display(po_val_summary)

PO header rows before join : 24,000
PO header rows after join  : 24,000  ✅ No fan-out

PO totals matching (±₹1)  : 24,000
PO totals mismatching     : 0


,check,total_orders,match,mismatch,status
0,PO header grand_total vs lines sum (...,24000,24000,0,✅ Pass


### 6-C  Invoices ↔ Payments — financial total reconciliation

In [73]:
# ── Parse dates ───────────────────────────────────────────────────────────────
for col in ["invoice_date","due_date"]:
    inv_clean[col] = pd.to_datetime(inv_clean[col], errors="coerce")
pay_clean["payment_date"] = pd.to_datetime(pay_clean["payment_date"], errors="coerce")

# ── Aggregate payments by invoice_id ─────────────────────────────────────────
# payments.invoice_id is the ONLY link from a payment to an invoice, and invoice_id
# is not unique in invoices (UI-01). A payment for a repeated invoice_id therefore
# cannot be attributed to one specific invoice row.
pay_agg_by_inv = (
    pay_clean
    .groupby("invoice_id", sort=False)
    .agg(
        payment_count       = ("row_key",        "count"),
        total_paid_amount   = ("payment_amount",  "sum"),
        first_payment_date  = ("payment_date",    "min"),
        last_payment_date   = ("payment_date",    "max"),
        payment_methods     = ("payment_method",  lambda x: " | ".join(x.unique()[:3])),
    )
    .reset_index()
)
print(f"Unique invoice_id in payments: {len(pay_agg_by_inv):,}")

# ── NAIVE join (kept only for root-cause analysis in 6-C follow-up) ──────────
# Every row sharing a repeated invoice_id receives the SAME total, so payments are
# counted once per row -> total paid is inflated.
inv_pay_naive = inv_clean.merge(pay_agg_by_inv, on="invoice_id", how="left")

# ── SAFE join (used downstream) ──────────────────────────────────────────────
# FIX: payment fields are populated only where invoice_id is unique, so each payment
# lands on exactly one invoice. Rows with repeated IDs get NaN + reliable=False
# instead of a wrong number.
inv_pay_joined = inv_pay_naive.copy()
inv_pay_joined["payment_total_reliable"] = ~inv_pay_joined["duplicate_invoice_id_flag"]
pay_cols = ["payment_count","total_paid_amount","first_payment_date","last_payment_date","payment_methods"]
inv_pay_joined.loc[~inv_pay_joined["payment_total_reliable"], pay_cols] = np.nan

# outstanding_amount only where payments are reliable (no negative balances from fan-out)
inv_pay_joined["outstanding_amount"] = np.where(
    inv_pay_joined["payment_total_reliable"],
    inv_pay_joined["grand_total"] - inv_pay_joined["total_paid_amount"].fillna(0),
    np.nan,
)

# ── Cardinality + financial-total checks ─────────────────────────────────────
print(f"Invoice rows before join: {len(inv_clean):,}")
print(f"Invoice rows after join : {len(inv_pay_joined):,}")
assert len(inv_pay_joined) == len(inv_clean), "Invoice-payment join caused fan-out!"
print("✅ No row multiplication from join")

naive_total = inv_pay_naive["total_paid_amount"].sum()
safe_total  = inv_pay_joined["total_paid_amount"].sum()
unique_ids_paid = pay_clean.loc[~pay_clean["invoice_id"].isin(
    set(inv_clean.loc[inv_clean["duplicate_invoice_id_flag"], "invoice_id"])), "payment_amount"].sum()
print(f"\nPayments file total                 : {pay_clean['payment_amount'].sum():>20,.2f}")
print(f"Naive join total_paid (inflated)    : {naive_total:>20,.2f}")
print(f"Safe join total_paid (reliable rows): {safe_total:>20,.2f}")
assert abs(safe_total - unique_ids_paid) < 1, "Safe join total must equal payments on unique invoice_ids"
assert naive_total > pay_clean["payment_amount"].sum(), "expected naive join to over-count"
print("✅ Safe join reconciles exactly to payments on unique invoice_ids")


Unique invoice_id in payments: 16,036
Invoice rows before join: 18,033
Invoice rows after join : 18,033
✅ No row multiplication from join

Payments file total                 :    23,308,879,670.97
Naive join total_paid (inflated)    :    23,827,028,417.07
Safe join total_paid (reliable rows):    22,795,333,903.87
✅ Safe join reconciles exactly to payments on unique invoice_ids


In [74]:
# ── Coverage & payment status analysis ───────────────────────────────────────
reliable = inv_pay_joined[inv_pay_joined["payment_total_reliable"]]
print(f"Reliable invoice rows (unique invoice_id): {len(reliable):,}  |  excluded (UI-01): {(~inv_pay_joined['payment_total_reliable']).sum()}")
invoices_with_payment  = reliable["total_paid_amount"].notna().sum()
invoices_no_payment    = reliable["total_paid_amount"].isna().sum()

print(f"Invoice rows with payment records : {invoices_with_payment:,}")
print(f"Invoice rows with no payment data : {invoices_no_payment:,}")

# Payment coverage by payment_status
coverage_by_status = (
    reliable
    .assign(has_payment=inv_pay_joined["total_paid_amount"].notna())
    .groupby(["payment_status","has_payment"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "no_payment_record", True: "has_payment_record"})
)
coverage_by_status["total"] = coverage_by_status.sum(axis=1)
print("\nPayment record coverage by invoice payment_status:")
display(coverage_by_status)

# ── Compare grand_total vs total_paid_amount for Paid invoices ───────────────
paid_invoices = reliable[
    (reliable["payment_status"] == "Paid") &
    (reliable["total_paid_amount"].notna())
].copy()

paid_invoices["payment_diff"]   = (
    paid_invoices["grand_total"] - paid_invoices["total_paid_amount"]
).round(2)
paid_invoices["payment_match"]  = paid_invoices["payment_diff"].abs() <= 1.0

print(f"\nPaid invoices with payment record  : {len(paid_invoices):,}")
print(f"grand_total ≈ paid_amount (±₹1)    : {paid_invoices['payment_match'].sum():,}")
print(f"grand_total ≠ paid_amount           : {(~paid_invoices['payment_match']).sum():,}")

# ── payment_status vs actual money (reliable rows only) ─────────────────────
r = reliable.assign(paid=reliable["total_paid_amount"].fillna(0))
status_checks = pd.DataFrame([
    {"check": "Paid but paid < grand_total",              "rows": int(((r.payment_status=="Paid")           & (r.paid < r.grand_total-1)).sum())},
    {"check": "Paid but paid > grand_total (overpaid)",   "rows": int(((r.payment_status=="Paid")           & (r.paid > r.grand_total+1)).sum())},
    {"check": "Unpaid but has payments",                  "rows": int(((r.payment_status=="Unpaid")         & (r.paid > 0)).sum())},
    {"check": "Partially Paid but fully paid",            "rows": int(((r.payment_status=="Partially Paid") & (r.paid >= r.grand_total-1)).sum())},
    {"check": "Partially Paid but no payments",           "rows": int(((r.payment_status=="Partially Paid") & (r.paid == 0)).sum())},
])
display(status_checks)
status_checks.to_csv(REPORTS_DIR / "task6c_payment_status_consistency.csv", index=False)


Reliable invoice rows (unique invoice_id): 17,640  |  excluded (UI-01): 393
Invoice rows with payment records : 15,842
Invoice rows with no payment data : 1,798

Payment record coverage by invoice payment_status:


has_payment,no_payment_record,has_payment_record,total
payment_status,,,
Paid,0,12264,12264
Partially Paid,0,3578,3578
Unpaid,1798,0,1798



Paid invoices with payment record  : 12,264
grand_total ≈ paid_amount (±₹1)    : 12,264
grand_total ≠ paid_amount           : 0


,check,rows
0,Paid but paid < grand_total,0
1,Paid but paid > grand_total (overpaid),0
2,Unpaid but has payments,0
3,Partially Paid but fully paid,0
4,Partially Paid but no payments,0


In [75]:
# ── Integration validation summary ───────────────────────────────────────────
integration_summary = pd.DataFrame([
    {
        "integration"   : "SO header ↔ SO lines (grand_total ±₹1)",
        "total_records" : len(so_integrated),
        "match"         : int(match_count),
        "mismatch"      : int(mismatch_count),
        "no_child_data" : int(null_lines),
        "status"        : "✅ Pass" if mismatch_count == 0 else "⚠️ Investigate",
    },
    {
        "integration"   : "PO header ↔ PO lines (grand_total ±₹1)",
        "total_records" : len(po_integrated),
        "match"         : int(po_match),
        "mismatch"      : int(po_mismatch),
        "no_child_data" : 0,
        "status"        : "✅ Pass" if po_mismatch == 0 else "⚠️ Investigate",
    },
    {
        "integration"   : "Invoice grand_total ≈ payment amount (Paid, ±₹1)",
        "total_records" : len(paid_invoices),
        "match"         : int(paid_invoices["payment_match"].sum()),
        "mismatch"      : int((~paid_invoices["payment_match"]).sum()),
        "no_child_data" : 0,
        "status"        : "✅ Pass" if (~paid_invoices["payment_match"]).sum() == 0 else "⚠️ Investigate",
    },
])

display(integration_summary)
integration_summary.to_csv(REPORTS_DIR / "task6_integration_validation_summary.csv", index=False)
print("Saved: task6_integration_validation_summary.csv")

audit_log.append({
    "task"             : "Task 6",
    "dataset"          : "SO, PO, invoices, payments",
    "action"           : "Integrated header-lines and invoices-payments with cardinality checks",
    "result"           : integration_summary["status"].to_list(),
    "records_affected" : 0,
    "decision"         : "No deletions. Mismatches documented for business review.",
})

,integration,total_records,match,mismatch,no_child_data,status
0,SO header ↔ SO lines (grand_total ±₹1),20000,20000,0,0,✅ Pass
1,PO header ↔ PO lines (grand_total ±₹1),24000,24000,0,0,✅ Pass
2,Invoice grand_total ≈ payment amount...,12264,12264,0,0,✅ Pass


Saved: task6_integration_validation_summary.csv


### 6-D  Date type coercions across all datasets

In [76]:
DATE_COLUMNS = {
    "purchase_orders_header" : ["order_date","expected_delivery_date","received_date"],
    "purchase_orders_lines"  : ["line_delivery_date"] if "line_delivery_date" in raw["purchase_orders_lines"].columns else [],
    "sales_orders_header"    : ["order_date","delivery_date"],
    "sales_orders_lines"     : ["line_delivery_date"] if "line_delivery_date" in raw["sales_orders_lines"].columns else [],
    "invoices"               : ["invoice_date","due_date"],
    "payments"               : ["payment_date"],
    "customers"              : ["customer_since","last_purchase_date"],
    "stock_ledger"           : ["movement_date"],
    "products"               : ["last_purchase_date"],
}

clean_datasets: dict[str, pd.DataFrame] = {}
coercion_log = []

for name, df in raw.items():
    cdf = df.copy()
    for col in DATE_COLUMNS.get(name, []):
        if col in cdf.columns:
            before_nulls = int(cdf[col].isna().sum())
            cdf[col]     = pd.to_datetime(cdf[col], errors="coerce")
            after_nulls  = int(cdf[col].isna().sum())
            new_nulls    = after_nulls - before_nulls
            coercion_log.append({
                "dataset"              : name,
                "column"               : col,
                "dtype_after"          : str(cdf[col].dtype),
                "nulls_before"         : before_nulls,
                "nulls_after"          : after_nulls,
                "new_nulls_from_parse" : new_nulls,
                "status"               : "✅ OK" if new_nulls == 0 else "⚠️ New nulls",
            })
    clean_datasets[name] = cdf

# Re-apply the earlier cleaning flags to the clean copies
clean_datasets["invoices"] = inv_clean.copy()
clean_datasets["payments"] = pay_clean.copy()

coercion_df = pd.DataFrame(coercion_log)
display(coercion_df)

coercion_df.to_csv(REPORTS_DIR / "task6_date_coercion_summary.csv", index=False)
print("Saved: task6_date_coercion_summary.csv")

audit_log.append({
    "task"             : "Task 6-D",
    "dataset"          : "multiple",
    "action"           : f"Parsed {len(coercion_df)} date columns to datetime64",
    "result"           : f"{(coercion_df['status']=='✅ OK').sum()}/{len(coercion_df)} columns parsed cleanly",
    "records_affected" : 0,
    "decision"         : "Dates stored as datetime64. New nulls (if any) logged for investigation.",
})

,dataset,column,dtype_after,nulls_before,nulls_after,new_nulls_from_parse,status
0,customers,customer_since,datetime64[us],0,0,0,✅ OK
1,customers,last_purchase_date,datetime64[us],0,0,0,✅ OK
2,invoices,invoice_date,datetime64[us],0,0,0,✅ OK
3,invoices,due_date,datetime64[us],0,0,0,✅ OK
4,payments,payment_date,datetime64[us],0,0,0,✅ OK
5,products,last_purchase_date,datetime64[us],0,0,0,✅ OK
6,purchase_orders_header,order_date,datetime64[us],0,0,0,✅ OK
7,purchase_orders_header,expected_delivery_date,datetime64[us],0,0,0,✅ OK
8,purchase_orders_header,received_date,datetime64[us],2370,2370,0,✅ OK
9,sales_orders_header,order_date,datetime64[us],0,0,0,✅ OK


Saved: task6_date_coercion_summary.csv


---
## Task 7 — Save Validation Results & Final Report
**Goal:** Export all cleaned datasets, write the audit log, build the before/after quality comparison, and produce the final Week-2 validation report.

In [77]:
# ── Before/after data-quality comparison ─────────────────────────────────────
ba_rows = []
for name in raw:
    raw_df   = raw[name]
    clean_df = clean_datasets[name]
    ba_rows.append({
        "dataset"                  : name,
        "rows_before"              : len(raw_df),
        "rows_after"               : len(clean_df),
        "row_delta"                : len(clean_df) - len(raw_df),
        "missing_cells_before"     : int(raw_df.isna().sum().sum()),
        "missing_cells_after"      : int(clean_df.isna().sum().sum()),
        "columns_before"           : raw_df.shape[1],
        "columns_after"            : clean_df.shape[1],
        "new_columns_added"        : clean_df.shape[1] - raw_df.shape[1],
    })

ba_df = pd.DataFrame(ba_rows)
display(ba_df)
ba_df.to_csv(REPORTS_DIR / "task7_before_after_quality_comparison.csv", index=False)
print("Saved: task7_before_after_quality_comparison.csv")

,dataset,rows_before,rows_after,row_delta,missing_cells_before,missing_cells_after,columns_before,columns_after,new_columns_added
0,branches,6,6,0,0,0,13,13,0
1,customers,500,500,0,0,0,15,15,0
2,inventory_master,180,180,0,0,0,8,8,0
3,invoices,18033,18033,0,0,0,10,12,2
4,payments,19257,19257,0,0,0,5,7,2
5,products,30,30,0,0,0,23,23,0
6,purchase_orders_header,24000,24000,0,2370,2370,10,10,0
7,purchase_orders_lines,155495,155495,0,0,0,9,9,0
8,sales_orders_header,20000,20000,0,0,0,11,11,0
9,sales_orders_lines,130402,130402,0,0,0,9,9,0


Saved: task7_before_after_quality_comparison.csv


In [78]:
# ── Export all cleaned datasets ───────────────────────────────────────────────
export_log = []
for name, cdf in clean_datasets.items():
    out_path = CLEAN_DIR / f"{name}_clean.csv"
    cdf.to_csv(out_path, index=False)
    export_log.append({
        "dataset"    : name,
        "rows"       : len(cdf),
        "columns"    : cdf.shape[1],
        "output_file": out_path.name,
    })

export_df = pd.DataFrame(export_log)
display(export_df)
export_df.to_csv(REPORTS_DIR / "task7_clean_export_manifest.csv", index=False)
print(f"\nAll {len(clean_datasets)} cleaned datasets written to: {CLEAN_DIR.resolve()}")

,dataset,rows,columns,output_file
0,branches,6,13,branches_clean.csv
1,customers,500,15,customers_clean.csv
2,inventory_master,180,8,inventory_master_clean.csv
3,invoices,18033,12,invoices_clean.csv
4,payments,19257,7,payments_clean.csv
5,products,30,23,products_clean.csv
6,purchase_orders_header,24000,10,purchase_orders_header_clean.csv
7,purchase_orders_lines,155495,9,purchase_orders_lines_clean.csv
8,sales_orders_header,20000,11,sales_orders_header_clean.csv
9,sales_orders_lines,130402,9,sales_orders_lines_clean.csv



All 12 cleaned datasets written to: /Users/macbook/Documents/Heavy-Supplier-Inventory-Warehouse-Analytics/week_02/reports/clean


In [79]:
# ── Save audit log ────────────────────────────────────────────────────────────
audit_df = pd.DataFrame(audit_log)
display(audit_df)
audit_df.to_csv(REPORTS_DIR / "task7_cleaning_audit_log.csv", index=False)
print("Saved: task7_cleaning_audit_log.csv")

,task,dataset,action,result,records_affected,decision
0,Task 1,all,Baseline row-count verification,All 12 datasets match Week-1 counts,0,No action — informational
1,Task 2,purchase_orders_header,Investigated missing received_date,100% of nulls belong to Cancelled PO...,2370,Retain all rows. Null received_date ...
2,Task 3-A,invoices,Investigated 196 repeated invoice_id...,All repeated IDs link to distinct tr...,393,Retain all rows. Added row_key surro...
3,Task 3-B,payments,Investigated 201 repeated payment_id...,All repeated IDs link to distinct tr...,403,Retain all rows. Added row_key surro...
4,Task 4,multiple,Validated 6 candidate primary keys +...,6/6 PKs valid; 17/17 FKs pass,0,No deletions. All failures documente...
5,Task 5,inventory_master + stock_ledger,Reconciled current_stock vs latest l...,14 mismatches found out of 180 pairs,14,Mismatches documented. No correction...
6,Task 6,"SO, PO, invoices, payments",Integrated header-lines and invoices...,"[✅ Pass, ✅ Pass, ✅ Pass]",0,No deletions. Mismatches documented ...
7,Task 6-D,multiple,Parsed 12 date columns to datetime64,12/12 columns parsed cleanly,0,Dates stored as datetime64. New null...


Saved: task7_cleaning_audit_log.csv


In [80]:
# ── Final validation checklist ────────────────────────────────────────────────
def check(label: str, passed: bool, detail: str = "") -> dict:
    return {"check": label, "passed": passed,
            "status": "✅ Pass" if passed else "❌ Fail", "detail": detail}

inv_mismatch_count  = int(reconcile["mismatch"].sum())
so_lines_mismatch   = int((~so_integrated["totals_match"]).sum())
po_lines_mismatch   = int((~po_integrated["totals_match"]).sum())
inv_pay_mismatch    = int((~paid_invoices["payment_match"]).sum())
fk_failures         = int((fk_df["status"] != "✅ Pass").sum())
pk_failures         = int((~pk_df["key_valid"]).sum())
date_parse_warnings = int((coercion_df["new_nulls_from_parse"] > 0).sum()) if len(coercion_df) else 0

checklist = [
    check("Task 1 — All 12 datasets loaded, row counts match Week-1",
          total_now == total_w1,
          f"{total_now:,} rows loaded"),
    check("Task 2 — Missing received_date fully explained by Cancelled status",
          len(received_null_in_received_status) == 0,
          "2,370 nulls = Cancelled POs only"),
    check("Task 3-A — Repeated invoice_id: no fully identical duplicate rows",
          len(fully_identical_inv) == 0,
          f"{len(inv_dup_detail)} IDs flagged; 0 identical duplicates"),
    check("Task 3-B — Repeated payment_id: no fully identical duplicate rows",
          len(fully_identical_pay) == 0,
          f"{len(pay_dup_detail)} IDs flagged; 0 identical duplicates"),
    check("Task 4-A — All candidate primary keys unique & non-null",
          pk_failures == 0,
          f"{pk_failures} key failures"),
    check("Task 4-B — All 17 FK relationships: zero unmatched references",
          fk_failures == 0,
          f"{fk_failures} FK failures"),
    check("Task 5 — Inventory balance reconciliation completed",
          True,          # Completed — mismatches may exist and are documented
          f"{inv_mismatch_count} product-branch pairs differ (documented)"),
    check("Task 6-A — SO grand_total matches SO lines sum (±₹1)",
          so_lines_mismatch == 0,
          f"{so_lines_mismatch} mismatches"),
    check("Task 6-B — PO grand_total matches PO lines sum (±₹1)",
          po_lines_mismatch == 0,
          f"{po_lines_mismatch} mismatches"),
    check("Task 6-C — Paid invoice total matches payment total (±₹1)",
          inv_pay_mismatch == 0,
          f"{inv_pay_mismatch} mismatches"),
    check("Task 6-D — Date columns parsed without new nulls",
          date_parse_warnings == 0,
          f"{date_parse_warnings} columns introduced new nulls"),
    check("Task 7 — All cleaned datasets exported",
          len(list(CLEAN_DIR.glob("*.csv"))) == len(raw),
          f"{len(list(CLEAN_DIR.glob('*.csv')))} files written"),
]

checklist_df = pd.DataFrame(checklist)
pass_count   = checklist_df["passed"].sum()
fail_count   = (~checklist_df["passed"]).sum()

display(checklist_df[["check","status","detail"]])
print(f"\n{'='*60}")
print(f"  Week-2 Validation: {pass_count} / {len(checklist)} checks passed")
print(f"{'='*60}")

checklist_df.to_csv(REPORTS_DIR / "task7_week2_validation_checklist.csv", index=False)
print("Saved: task7_week2_validation_checklist.csv")

,check,status,detail
0,"Task 1 — All 12 datasets loaded, row...",✅ Pass,"605,141 rows loaded"
1,Task 2 — Missing received_date fully...,✅ Pass,"2,370 nulls = Cancelled POs only"
2,Task 3-A — Repeated invoice_id: no f...,✅ Pass,196 IDs flagged; 0 identical duplicates
3,Task 3-B — Repeated payment_id: no f...,✅ Pass,201 IDs flagged; 0 identical duplicates
4,Task 4-A — All candidate primary key...,✅ Pass,0 key failures
5,Task 4-B — All 17 FK relationships: ...,✅ Pass,0 FK failures
6,Task 5 — Inventory balance reconcili...,✅ Pass,14 product-branch pairs differ (docu...
7,Task 6-A — SO grand_total matches SO...,✅ Pass,0 mismatches
8,Task 6-B — PO grand_total matches PO...,✅ Pass,0 mismatches
9,Task 6-C — Paid invoice total matche...,✅ Pass,0 mismatches



  Week-2 Validation: 12 / 12 checks passed
Saved: task7_week2_validation_checklist.csv


In [81]:
# ── Unresolved issues requiring business-rule confirmation ────────────────────
unresolved = [
    {
        "issue_id"   : "UI-01",
        "area"       : "Invoices",
        "description": (
            "196 invoice_id values shared across distinct transactions. "
            "Source system must confirm intended uniqueness rule and whether "
            "re-numbering is possible without disrupting downstream references."
        ),
        "impact"     : "Financial aggregation by invoice_id will double-count if not controlled.",
        "mitigation" : "Payments joined only to unique invoice_ids; repeated IDs carry NaN payments and payment_total_reliable=False. Row_key alone cannot attribute payments (payments only reference invoice_id).",
        "owner"      : "Finance / IT",
    },
    {
        "issue_id"   : "UI-02",
        "area"       : "Payments",
        "description": (
            "201 payment_id values shared across distinct transactions. "
            "Payment ID generation logic must be reviewed in source system."
        ),
        "impact"     : "Payment totals by payment_id will be incorrect if IDs are treated as unique.",
        "mitigation" : "Use row_key surrogate; filter on duplicate_payment_id_flag where needed.",
        "owner"      : "Finance / IT",
    },
    {
        "issue_id"   : "UI-03",
        "area"       : "Inventory",
        "description": (
            f"{inv_mismatch_count} (product_id, branch_id) pairs show a difference between "
            "inventory_master.current_stock and the latest stock_ledger running_balance. "
            "Business must confirm which is the authoritative balance figure."
        ),
        "impact"     : "Inventory valuation and reorder calculations may use incorrect balances.",
        "mitigation" : "Ledger running_balance chains correctly in file order (only ADJUSTMENT rows carry an unsigned quantity). Use ledger for time series; flag the mismatching pairs; confirm ADJUSTMENT sign convention.",
        "owner"      : "Warehouse / Operations",
    },
    {
        "issue_id"   : "UI-04",
        "area"       : "Purchase Orders",
        "description": (
            "2,370 purchase orders have NULL received_date with status='Cancelled'. "
            "Business confirmation required that Cancelled POs are never expected to have "
            "a received_date (currently treated as expected by design)."
        ),
        "impact"     : "Low — decision treats null as valid; lead-time analysis excludes Cancelled POs.",
        "mitigation" : "Document business rule and exclude Cancelled POs from delivery KPI calculations.",
        "owner"      : "Procurement",
    },
]

unresolved_df = pd.DataFrame(unresolved)
display(unresolved_df[["issue_id","area","description","impact","mitigation","owner"]])

unresolved_df.to_csv(REPORTS_DIR / "task7_unresolved_issues.csv", index=False)
print(f"\nSaved: task7_unresolved_issues.csv ({len(unresolved_df)} open issues)")

,issue_id,area,description,impact,mitigation,owner
0,UI-01,Invoices,196 invoice_id values shared across ...,Financial aggregation by invoice_id ...,Payments joined only to unique invoi...,Finance / IT
1,UI-02,Payments,201 payment_id values shared across ...,Payment totals by payment_id will be...,Use row_key surrogate; filter on dup...,Finance / IT
2,UI-03,Inventory,"14 (product_id, branch_id) pairs sho...",Inventory valuation and reorder calc...,Ledger running_balance chains correc...,Warehouse / Operations
3,UI-04,Purchase Orders,"2,370 purchase orders have NULL rece...",Low — decision treats null as valid;...,Document business rule and exclude C...,Procurement



Saved: task7_unresolved_issues.csv (4 open issues)


### 6-C (Follow-up) — Root Cause of Paid Invoice vs Payment Mismatches

**Expected:** `invoices.grand_total ≈ SUM(payments.payment_amount)` for each Paid invoice.  
**Found:** 247 mismatches where the aggregated payment total ≠ invoice total.

**Hypothesis:** Because `invoice_id` is not unique in `invoices`, aggregating payments
by `invoice_id` collapses payments for **multiple distinct transactions** into one total,
which is then compared against just one invoice’s `grand_total` — causing an apparent mismatch.

In [82]:
# ── Root-cause analysis of the 247 paid-invoice mismatches ───────────────
# Hypothesis: all mismatches are in rows where duplicate_invoice_id_flag=True.
# When invoice_id appears N times in invoices, pay_agg_by_inv returns the
# SUM of payments for ALL N invoices sharing that ID, inflating apparent paid total.

# Root-cause uses the NAIVE join (inv_pay_naive), which is where the mismatches arise.
naive_paid = inv_pay_naive[(inv_pay_naive["payment_status"] == "Paid") & inv_pay_naive["total_paid_amount"].notna()].copy()
naive_paid["payment_diff"]  = (naive_paid["grand_total"] - naive_paid["total_paid_amount"]).round(2)
naive_paid["payment_match"] = naive_paid["payment_diff"].abs() <= 1.0
paid_invoices_naive = naive_paid
dup_flags = naive_paid.loc[~naive_paid["payment_match"], "duplicate_invoice_id_flag"]
print(f"Naive-join Paid mismatches: {(~naive_paid['payment_match']).sum()}  |  after safe join: {(~paid_invoices['payment_match']).sum()}")
print(f"Mismatched rows with duplicate_invoice_id_flag=True : {dup_flags.sum()}")
print(f"Mismatched rows with duplicate_invoice_id_flag=False: {(~dup_flags).sum()}")
print()
print("CONCLUSION: 100% of payment mismatches are caused by UI-01.")
print("Root cause: aggregating payments by a non-unique invoice_id sums payments")
print("across all transactions sharing that ID, inflating the per-row total.")
print()

# ── DECISION ──────────────────────────────────────────────────────────────
# Do NOT use invoice_id as the join key for payment reconciliation
# until the ID uniqueness issue (UI-01) is resolved in the source system.
# FIX APPLIED in 6-C: payments are attached only to unique invoice_ids; repeated IDs are NaN + payment_total_reliable=False.

# Save the mismatch detail report
mismatch_detail = naive_paid[~naive_paid["payment_match"]].copy()
save_cols = [
    "invoice_id", "so_id", "customer_id", "grand_total",
    "total_paid_amount", "payment_diff",
    "duplicate_invoice_id_flag", "payment_count"
]
mismatch_detail[save_cols].reset_index(drop=True).to_csv(
    REPORTS_DIR / "task6c_paid_invoice_payment_mismatches.csv", index=False
)
print(f"Saved: task6c_paid_invoice_payment_mismatches.csv ({len(mismatch_detail)} rows)")
print()
display(mismatch_detail[save_cols].head(10).reset_index(drop=True))

audit_log.append({
    "task"             : "Task 6-C (Follow-up)",
    "dataset"          : "invoices + payments",
    "action"           : "Root-cause analysis of 247 paid-invoice payment mismatches",
    "result"           : "100% caused by non-unique invoice_id (UI-01). Not a payment data error.",
    "records_affected" : 247,
    "decision"         : "FIXED: payment totals attached only to unique invoice_ids; repeated IDs flagged payment_total_reliable=False.",
})

Naive-join Paid mismatches: 247  |  after safe join: 0
Mismatched rows with duplicate_invoice_id_flag=True : 247
Mismatched rows with duplicate_invoice_id_flag=False: 0

CONCLUSION: 100% of payment mismatches are caused by UI-01.
Root cause: aggregating payments by a non-unique invoice_id sums payments
across all transactions sharing that ID, inflating the per-row total.

Saved: task6c_paid_invoice_payment_mismatches.csv (247 rows)



,invoice_id,so_id,customer_id,grand_total,total_paid_amount,payment_diff,duplicate_invoice_id_flag,payment_count
0,INV-361583,SO-624730,C0285,"905,589.00","2,282,421.40","-1,376,832.40",True,2.00
1,INV-166804,SO-975444,C0480,"1,634,013.00","3,537,163.16","-1,903,150.16",True,2.00
2,INV-603462,SO-619114,C0157,"931,608.60","1,540,888.60","-609,280.00",True,2.00
3,INV-632936,SO-432598,C0159,"3,085,330.60","5,271,746.60","-2,186,416.00",True,3.00
4,INV-695723,SO-836461,C0067,"202,560.00","4,485,952.40","-4,283,392.40",True,2.00
5,INV-414608,SO-130872,C0092,"3,195,330.40","6,993,193.40","-3,797,863.00",True,4.00
6,INV-850022,SO-964455,C0220,"205,504.00","1,113,914.60","-908,410.60",True,3.00
7,INV-173978,SO-690067,C0335,"2,541,624.00","3,385,385.40","-843,761.40",True,2.00
8,INV-112235,SO-363636,C0198,"3,023,775.00","4,909,097.80","-1,885,322.80",True,2.00
9,INV-567842,SO-911289,C0124,"513,231.20","841,921.20","-328,690.00",True,2.00


In [83]:
# ── Final: list all report files produced ────────────────────────────────────
all_report_files = sorted(REPORTS_DIR.rglob("*.csv"))
print(f"Week-2 report files generated: {len(all_report_files)}")
print()
for f in all_report_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.relative_to(REPORTS_DIR)}  ({size_kb:.1f} KB)")

Week-2 report files generated: 29

  clean/branches_clean.csv  (0.8 KB)
  clean/customers_clean.csv  (57.1 KB)
  clean/inventory_master_clean.csv  (6.6 KB)
  clean/invoices_clean.csv  (1852.6 KB)
  clean/payments_clean.csv  (1175.8 KB)
  clean/products_clean.csv  (4.6 KB)
  clean/purchase_orders_header_clean.csv  (2521.4 KB)
  clean/purchase_orders_lines_clean.csv  (10575.4 KB)
  clean/sales_orders_header_clean.csv  (2042.5 KB)
  clean/sales_orders_lines_clean.csv  (6904.2 KB)
  clean/stock_ledger_clean.csv  (14532.0 KB)
  clean/suppliers_clean.csv  (1.0 KB)
  task1_baseline_comparison.csv  (0.5 KB)
  task2_received_date_status_analysis.csv  (0.1 KB)
  task3a_invoice_id_duplicate_analysis.csv  (4.9 KB)
  task3b_payment_id_duplicate_analysis.csv  (4.2 KB)
  task4a_primary_key_validation.csv  (0.4 KB)
  task4b_foreign_key_validation.csv  (1.2 KB)
  task5_inventory_balance_mismatches.csv  (0.8 KB)
  task5_inventory_balance_reconciliation.csv  (8.9 KB)
  task6_date_coercion_summary.csv  (0

---
## Week-2 Findings Summary

| Task | Finding | Decision | Status |
|---|---|---|---|
| 1 — Load datasets | All 12 loaded; **605,141 rows** match Week-1 exactly | — | ✅ Complete |
| 2 — Missing dates | 2,370 nulls — **100% Cancelled POs**; 0 Received POs affected | Retain; null valid for Cancelled | ✅ Resolved |
| 3-A — Invoice IDs | 196 repeated IDs; **0 fully identical rows**; each links a distinct SO/customer/date | Flag with `duplicate_invoice_id_flag` | ⚠️ UI-01 pending |
| 3-B — Payment IDs | 201 repeated IDs; **0 fully identical rows**; each links distinct invoice | Flag with `duplicate_payment_id_flag` | ⚠️ UI-02 pending |
| 4 — Keys & FKs | All 6 PKs unique & non-null; all **17 FK checks pass** (0 unmatched) | — | ✅ Complete |
| 5 — Inventory | **14** product-branch pairs: `current_stock` ≠ ledger `running_balance` | Document; ledger chain valid in file order; confirm ADJUSTMENT signs | ⚠️ UI-03 open |
| 6-A SO | All **20,000** SO `grand_total` values match line sums within ±₹1 | — | ✅ Complete |
| 6-B PO | All **24,000** PO `grand_total` values match line sums within ±₹1 | — | ✅ Complete |
| 6-C Inv-Pay | **247** paid-invoice totals ≠ aggregated payment — **root cause = UI-01** | Avoid joining on non-unique `invoice_id` | ❌ Root cause = UI-01 |
| 6-D Dates | 12 date columns parsed to `datetime64`; **0 new nulls** introduced | — | ✅ Complete |
| 7 — Save | 28 CSV files + 12 cleaned datasets + audit log written | — | ✅ Complete |

---

### Open Issues (Require Business Confirmation)

| ID | Area | Issue | Impact | Owner |
|---|---|---|---|---|
| UI-01 | Invoices | 196 `invoice_id` values shared across distinct transactions | Financial aggregation double-counts; drives UI-05 | Finance / IT |
| UI-02 | Payments | 201 `payment_id` values shared across distinct transactions | Payment totals incorrect if ID treated as unique | Finance / IT |
| UI-03 | Inventory | 14 product-branch pairs: `current_stock` ≠ ledger balance | Wrong stock levels in reorder/valuation | Warehouse / Ops |
| UI-04 | Purchase Orders | 2,370 null `received_date` for Cancelled POs (treated as by-design) | Exclude from lead-time KPIs | Procurement |
| UI-05 | Inv–Pay match | 247 Paid invoice totals ≠ payment totals (caused entirely by UI-01) | Financial reconciliation unreliable until UI-01 resolved | Finance |

---

### Mitigation Until Issues Are Resolved
- Payments are joined **only to unique `invoice_id`s**; repeated IDs have NaN payments and `payment_total_reliable = False` (`row_key` cannot attribute payments).
- Filter on `duplicate_invoice_id_flag` / `duplicate_payment_id_flag` to isolate affected records.
- Use **`stock_ledger.running_balance`** (latest per product-branch) for inventory calculations.
- Exclude `po_status = Cancelled` from all delivery and lead-time KPI calculations.

---

### Output Files
| Location | Contents |
|---|---|
| `reports/week_02/clean/` | 12 cleaned datasets with `datetime64` types and flag columns |
| `task1_baseline_comparison.csv` | Row-count vs Week-1 baseline |
| `task2_received_date_status_analysis.csv` | PO status × received_date cross-tab |
| `task3a_invoice_id_duplicate_analysis.csv` | 196 repeated invoice IDs with diversity stats |
| `task3b_payment_id_duplicate_analysis.csv` | 201 repeated payment IDs with diversity stats |
| `task4a_primary_key_validation.csv` | PK uniqueness + null check (6 tables) |
| `task4b_foreign_key_validation.csv` | FK referential integrity (17 relationships) |
| `task5_inventory_balance_reconciliation.csv` | Full 180-pair balance comparison |
| `task5_inventory_balance_mismatches.csv` | 14 mismatch detail rows |
| `task6_integration_validation_summary.csv` | SO/PO/invoice integration results |
| `task6_date_coercion_summary.csv` | Date column parse results (12 columns) |
| `task6c_paid_invoice_payment_mismatches.csv` | 247 root-cause mismatch detail |
| `task7_before_after_quality_comparison.csv` | Before/after metrics for all 12 datasets |
| `task7_clean_export_manifest.csv` | Export manifest for clean/ folder |
| `task7_cleaning_audit_log.csv` | Full decision log for every cleaning action |
| `task7_week2_validation_checklist.csv` | 12-item pass/fail checklist |
| `task7_unresolved_issues.csv` | 5 open issues with owner and mitigation |

**Next:** Week 3 — Analytical layer: supplier performance, inventory turnover, payment aging, and reorder alerts.

---
## Task 8 — Ledger integrity & corrected merged datasets
Adds the checks and derived-field fixes found when the merged files were reviewed:
1. Ledger `running_balance` chain test (file order vs the wrong `movement_id` tie-break).
2. Rebuild of the 5 merged datasets with **correct `line_profit`** and **safe payment joins**.

In [84]:
# ── Ledger chain test: previous running_balance + signed movement == running_balance ──
lg = raw["stock_ledger"].copy()
lg["movement_date"] = pd.to_datetime(lg["movement_date"], errors="coerce")
lg["ledger_pos"]    = np.arange(len(lg))
KEY = ["product_id","branch_id"]
lg["signed_qty"] = np.where(lg["movement_type"] == "OUT", -lg["quantity"], lg["quantity"])

def chain_breaks(df, order):
    d = df.sort_values(order, kind="stable").copy()
    d["prev"] = d.groupby(KEY)["running_balance"].shift()
    d = d.dropna(subset=["prev"])
    return d[(d["prev"] + d["signed_qty"]) != d["running_balance"]]

b_pos = chain_breaks(lg, KEY + ["movement_date","ledger_pos"])
b_id  = chain_breaks(lg, KEY + ["movement_date","movement_id"])
print(f"Chain breaks — file order        : {len(b_pos):,}   by type: {b_pos['movement_type'].value_counts().to_dict()}")
print(f"Chain breaks — movement_id order : {len(b_id):,}   (movement_id is NOT chronological; do not sort by it)")

adj = lg[lg["movement_type"] == "ADJUSTMENT"]
print(f"\nADJUSTMENT rows: {len(adj):,} — quantity is unsigned in the source, but ~half decrease the balance.")
print("→ Non-ADJUSTMENT breaks:", int((b_pos['movement_type'] != 'ADJUSTMENT').sum()), "(must be 0)")
assert (b_pos["movement_type"] != "ADJUSTMENT").sum() == 0

audit_log.append({
    "task":"Task 8-A","dataset":"stock_ledger","action":"Ledger running_balance chain test",
    "result":f"{len(b_pos)} breaks in file order, all ADJUSTMENT (sign lost); {len(b_id)} breaks if sorted by movement_id",
    "records_affected":int(len(b_pos)),
    "decision":"Order by (date, file position). Treat ADJUSTMENT delta as running_balance - previous running_balance; confirm sign convention with Warehouse.",
})

Chain breaks — file order        : 1,222   by type: {'ADJUSTMENT': 1222}
Chain breaks — movement_id order : 56,583   (movement_id is NOT chronological; do not sort by it)

ADJUSTMENT rows: 2,454 — quantity is unsigned in the source, but ~half decrease the balance.
→ Non-ADJUSTMENT breaks: 0 (must be 0)


In [85]:
# ── Rebuild merged datasets with fixes ───────────────────────────────────────
MERGED_DIR = REPORTS_DIR / "merged"; MERGED_DIR.mkdir(exist_ok=True)
prod = clean_datasets["products"]; br = clean_datasets["branches"]; cu = clean_datasets["customers"]; su = clean_datasets["suppliers"]
sh = clean_datasets["sales_orders_header"]; po_h = clean_datasets["purchase_orders_header"]
def one_to_one(left, right, on, name, **kw):
    n = len(left); out = left.merge(right, on=on, how="left", validate="many_to_one", **kw)
    assert len(out) == n, f"{name}: fan-out"; return out

# SALES — line grain
so = so_lines.merge(sh, on="so_id", how="left", validate="many_to_one")
so = one_to_one(so, prod[["product_id","product_name","category","machine_type","brand","uom","unit_cost","margin_percentage","criticality_level"]], "product_id", "sales/product")
so = one_to_one(so, cu[["customer_id","customer_type","industry_segment","city","state","region","payment_terms","customer_rating"]]
                .rename(columns={"city":"customer_city","state":"customer_state","region":"customer_region","payment_terms":"customer_payment_terms"}), "customer_id", "sales/customer")
so = one_to_one(so, br[["branch_id","branch_name","city","state","region","warehouse_type"]]
                .rename(columns={"city":"branch_city","state":"branch_state","region":"branch_region"}), "branch_id", "sales/branch")
# FIX: margin_percentage in products is a MARKUP on cost, so profit must come from price − cost
so["line_profit"]  = (so["quantity"] * (so["unit_price"] - so["unit_cost"])).round(2)
so["line_margin_pct"] = np.where(so["line_total"] != 0, so["line_profit"] / so["line_total"] * 100, np.nan)
so["days_to_deliver"] = (so["delivery_date"] - so["order_date"]).dt.days
assert len(so) == len(so_lines)

# PURCHASE ORDERS — line grain
po = po_lines.merge(po_h, on="po_id", how="left", validate="many_to_one")
po = one_to_one(po, prod[["product_id","product_name","category","machine_type","brand","uom","unit_price","margin_percentage","criticality_level","lead_time_days"]], "product_id", "po/product")
po = one_to_one(po, su[["supplier_id","supplier_name","supplier_type","product_category","city","province","region","lead_time_days","reliability_score"]]
                .rename(columns={"city":"supplier_city","province":"supplier_province","region":"supplier_region","lead_time_days":"supplier_lead_days"}), "supplier_id", "po/supplier")
po = one_to_one(po, br[["branch_id","branch_name","city","state","region"]]
                .rename(columns={"city":"branch_city","state":"branch_state","region":"branch_region"}), "branch_id", "po/branch")
po["delivery_delay_days"]   = (po["received_date"] - po["expected_delivery_date"]).dt.days
po["lead_time_actual_days"] = (po["received_date"] - po["order_date"]).dt.days
assert len(po) == len(po_lines)

# INVOICES — safe payment join (see 6-C)
inv_m = inv_pay_joined.copy()
inv_m = one_to_one(inv_m, cu[["customer_id","customer_type","industry_segment","city","state","region","customer_rating"]]
                   .rename(columns={"city":"customer_city","state":"customer_state","region":"customer_region"}), "customer_id", "inv/customer")
inv_m = one_to_one(inv_m, br[["branch_id","branch_name","city","region"]].rename(columns={"city":"branch_city"}), "branch_id", "inv/branch")
inv_m = one_to_one(inv_m, sh[["so_id","order_date","delivery_date","order_status","sales_channel","payment_terms"]], "so_id", "inv/so")
AS_OF = pd.Timestamp(clean_datasets["invoices"]["due_date"].max())   # explicit, visible reporting date
inv_m["as_of_date"]   = AS_OF
unpaid_rel = inv_m["payment_total_reliable"] & (inv_m["outstanding_amount"] > 0.5)
inv_m["is_overdue"]   = unpaid_rel & (inv_m["due_date"] < AS_OF)
inv_m["days_overdue"] = np.where(inv_m["is_overdue"], (AS_OF - inv_m["due_date"]).dt.days, 0)
assert len(inv_m) == len(inv_clean)
assert not (inv_m["outstanding_amount"] < -0.5).any(), "negative outstanding should be impossible now"

# INVENTORY — ledger position, not movement_id
tot = (lg.pivot_table(index=KEY, columns="movement_type", values="quantity", aggfunc="sum", fill_value=0)
         .rename(columns={"IN":"total_in_qty","OUT":"total_out_qty","ADJUSTMENT":"total_adjustment_qty"}).reset_index())
inv_l = one_to_one(inv_master, latest_balance, KEY, "inv/ledger")
inv_l = one_to_one(inv_l, tot, KEY, "inv/totals")
inv_l = one_to_one(inv_l, prod[["product_id","product_name","category","unit_cost","unit_price"]], "product_id", "inv/product")
inv_l["balance_difference"] = inv_l["current_stock"] - inv_l["ledger_latest_balance"]
inv_l["balance_mismatch"]   = inv_l["balance_difference"] != 0
inv_l["stock_value_cost"]   = inv_l["current_stock"] * inv_l["unit_cost"]
inv_l["below_reorder"]      = inv_l["current_stock"] < inv_l["reorder_level"]

# LEDGER — enriched
led_e = one_to_one(lg.drop(columns=["ledger_pos","signed_qty"]), prod[["product_id","product_name","category","unit_cost","unit_price"]], "product_id", "ledger/product")
led_e["movement_value_cost"]  = led_e["quantity"] * led_e["unit_cost"]
led_e["movement_value_price"] = led_e["quantity"] * led_e["unit_price"]

merged = {"sales_orders_full":so, "purchase_orders_full":po, "invoices_with_payments":inv_m,
          "inventory_with_ledger":inv_l, "stock_ledger_enriched":led_e}
manifest = []
for n, d in merged.items():
    d.to_csv(MERGED_DIR / f"{n}.csv", index=False); manifest.append({"file":f"{n}.csv","rows":len(d),"columns":d.shape[1]})
manifest = pd.DataFrame(manifest); manifest.to_csv(MERGED_DIR / "merged_datasets_manifest.csv", index=False); display(manifest)

# ── Validation of the merged outputs ─────────────────────────────────────────
tests = pd.DataFrame([
    {"test":"sales rows == SO lines",                         "ok": len(so)==len(so_lines)},
    {"test":"PO rows == PO lines",                            "ok": len(po)==len(po_lines)},
    {"test":"invoice rows == invoices",                       "ok": len(inv_m)==len(inv_clean)},
    {"test":"sales line_total sum == SO lines sum",           "ok": abs(so.line_total.sum()-so_lines.line_total.sum())<1},
    {"test":"line_profit == qty*(price-cost) for every row",  "ok": bool(((so.quantity*(so.unit_price-so.unit_cost)-so.line_profit).abs()<=0.01).all())},
    {"test":"sales margin % is plausible (<= 100)",           "ok": bool((so.line_margin_pct<=100).all())},
    {"test":"no negative outstanding_amount",                 "ok": not bool((inv_m.outstanding_amount<-0.5).any())},
    {"test":"merged total_paid <= payments file total",       "ok": inv_m.total_paid_amount.sum() <= pay_clean.payment_amount.sum()+1},
    {"test":"is_overdue implies unpaid & reliable",           "ok": bool((~inv_m.is_overdue | (inv_m.outstanding_amount>0.5)).all())},
])
display(tests); assert tests["ok"].all(), "merged validation failed"
print(f"Reported line profit: {so.line_profit.sum():,.0f}  ({so.line_profit.sum()/so.line_total.sum():.1%} margin)")
audit_log.append({"task":"Task 8-B","dataset":"merged outputs","action":"Rebuilt 5 merged datasets with fixed line_profit and safe payment join",
                  "result":"all validation tests passed","records_affected":int(len(so)),"decision":"Replace previous merged files."})

,file,rows,columns
0,sales_orders_full.csv,130402,42
1,purchase_orders_full.csv,155495,41
2,invoices_with_payments.csv,18033,36
3,inventory_with_ledger.csv,180,22
4,stock_ledger_enriched.csv,237230,15


,test,ok
0,sales rows == SO lines,True
1,PO rows == PO lines,True
2,invoice rows == invoices,True
3,sales line_total sum == SO lines sum,True
4,line_profit == qty*(price-cost) for ...,True
5,sales margin % is plausible (<= 100),True
6,no negative outstanding_amount,True
7,merged total_paid <= payments file t...,True
8,is_overdue implies unpaid & reliable,True


Reported line profit: 5,485,544,390  (21.6% margin)
